In [12]:
import os
import zipfile
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch.nn.functional as F

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {device}")
if device.type == 'cpu':
    print(" WARNING: You are using CPU. Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU")

🖥️ Using device: cuda


In [18]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
# Adjust these paths if your uploaded zip files have different names!
TRAIN_ZIP = '/content/train_data.zip'
TEST_ZIP = '/content/my_test_images.zip'
TRAIN_CSV = '/content/Train.xlsx'  # Upload this directly to /content/

TRAIN_DIR = '/content/train_data'
TEST_DIR = '/content/test_data'

# The optimal threshold found during your ablation study
THRESHOLD = 0.45

# ==========================================
# 2. UNZIP DATA SAFELY
# ==========================================
def extract_zip(zip_path, extract_to):
    if os.path.exists(zip_path):
        if not os.path.exists(extract_to):
            print(f" Extracting {zip_path}...")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_to)
            print(" Extraction complete.")
        else:
            print(f" {extract_to} already exists. Skipping extraction.")
    else:
        print(f" ERROR: Cannot find {zip_path}. Please upload it to Colab!")

extract_zip(TRAIN_ZIP, TRAIN_DIR)
extract_zip(TEST_ZIP, TEST_DIR)

# Find the exact subfolder containing the training images
try:
    # This searches for a folder that contains lots of images
    TRAIN_IMG_DIR = [r for r, d, f in os.walk(TRAIN_DIR) if len(f) > 100][0]
    print(f" Found training images at: {TRAIN_IMG_DIR}")
except IndexError:
    print(" ERROR: Could not find the training images inside the unzipped folder.")

✅ /content/train_data already exists. Skipping extraction.
📦 Extracting /content/my_test_images.zip...
✅ Extraction complete.
📂 Found training images at: /content/train_data/train_data


In [26]:
# ==========================================
# 3. CUSTOM DATASET & AUGMENTATIONS (Updated for Excel)
# ==========================================
class NakbaDataset(Dataset):
    def __init__(self, file_path, img_dir, transform=None):
        #
        self.img_labels = pd.read_excel(file_path)
        self.img_dir = img_dir
        self.transform = transform
        self.label_map = {'not_destruction': 0, 'destruction': 1}

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        # Assumes image names are in column 0 and labels in column 1
        img_name = self.img_labels.iloc[idx, 0]
        label_str = self.img_labels.iloc[idx, 1]
        img_path = os.path.join(self.img_dir, img_name)

        try:
             image = Image.open(img_path).convert("RGB")
        except Exception:
             # Failsafe for corrupted or missing images
             image = Image.new('RGB', (256, 256), color='black')

        label = self.label_map.get(label_str, 0)

        if self.transform:
            image = self.transform(image)

        return image, label

#
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.Grayscale(num_output_channels=3), # Kills color bias (sepia/B&W)
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print(" Dataset and Augmentations Configured (Ready for Excel).")

✅ Dataset and Augmentations Configured (Ready for Excel).


In [27]:
# Make sure you set this at the top of your notebook!
TRAIN_FILE = '/content/Train.xlsx'

if not os.path.exists(TRAIN_FILE):
    raise FileNotFoundError(f" Please upload {TRAIN_FILE} to Colab!")

# We pass the Excel file to the dataset
train_dataset = NakbaDataset(file_path=TRAIN_FILE, img_dir=TRAIN_IMG_DIR, transform=train_transform)

# Calculate weights for CrossEntropyLoss to handle class imbalance
labels_in_file = pd.read_excel(TRAIN_FILE).iloc[:, 1]
class_counts = labels_in_file.value_counts()
print("\n Training Data Distribution:")
print(class_counts)

num_not_destruction = class_counts.get('not_destruction', 1)
num_destruction = class_counts.get('destruction', 1)

# Weights = 1 / Frequency
weights = [1.0 / num_not_destruction, 1.0 / num_destruction]
class_weights = torch.FloatTensor(weights).to(device)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
print(" DataLoader ready.")


📊 Training Data Distribution:
label
not_destruction    906
destruction        494
Name: count, dtype: int64
✅ DataLoader ready.


In [28]:
# Model Initialization & Training
print("Initializing ResNet-50...")
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

epochs = 6
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss(weight=class_weights)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

print("Starting Training Loop...")
model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()
    print(f"    Epoch {epoch+1}/{epochs} Complete - Avg Loss: {running_loss/len(train_loader):.4f}")

# Save the best weights
torch.save(model.state_dict(), 'best_model.pth')
print(" Training Complete. Model saved.")

🚀 Initializing ResNet-50...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 178MB/s]


🔥 Starting Training Loop...
   ⭐ Epoch 1/6 Complete - Avg Loss: 0.5431
   ⭐ Epoch 2/6 Complete - Avg Loss: 0.3929
   ⭐ Epoch 3/6 Complete - Avg Loss: 0.2702
   ⭐ Epoch 4/6 Complete - Avg Loss: 0.2063
   ⭐ Epoch 5/6 Complete - Avg Loss: 0.1268
   ⭐ Epoch 6/6 Complete - Avg Loss: 0.1110
✅ Training Complete. Model saved.


In [30]:
# Test-Time Augmentation (TTA) Inference
print(" Running TTA Inference on Test Set...")
model.eval()

# Three different perspectives of the test image
tta_transforms = [
    transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((256, 256)), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
]

# Find where test images actually are
try:
    ACTUAL_TEST_DIR = [r for r, d, f in os.walk(TEST_DIR) if len(f) > 10][0]
except IndexError:
    ACTUAL_TEST_DIR = TEST_DIR # Fallback

results = []
DESTRUCTION_IDX = 1 # Because 'destruction' maps to 1 in our dataset class

for img_name in os.listdir(ACTUAL_TEST_DIR):
    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(ACTUAL_TEST_DIR, img_name)
        try:
            image = Image.open(img_path).convert("RGB")

            tta_probs = []
            with torch.no_grad():
                for t in tta_transforms:
                    tensor = t(image).unsqueeze(0).to(device)
                    probs = F.softmax(model(tensor), dim=1)
                    tta_probs.append(probs[0][DESTRUCTION_IDX].item())

            # Average predictions across the 3 augmented crops
            final_prob = sum(tta_probs) / len(tta_probs)

            # Apply our Custom Threshold
            label = "destruction" if final_prob >= THRESHOLD else "not_destruction"
            results.append({'name': img_name, 'label': label})
        except Exception as e:
            print(f" Error predicting {img_name}: {e}")

# ==========================================
# 7. EXPORT TO CSV AND ZIP
# ==========================================
if len(results) > 0:
    final_df = pd.DataFrame(results)
    print("\n FINAL Predictions Distribution:")
    print(final_df['label'].value_counts())

    final_df.to_csv('predictions.csv', index=False)
    with zipfile.ZipFile('winning_submission.zip', 'w') as z:
        z.write('predictions.csv')

    print(" Boom! 'winning_submission.zip' is ready for download.")
    try:
        from google.colab import files
        files.download('winning_submission.zip')
    except: pass
else:
    print(" No test images found. Check your test zip file.")

⏳ Running TTA Inference on Test Set...

📊 FINAL Predictions Distribution:
label
not_destruction    361
destruction         41
Name: count, dtype: int64
🎉 Boom! 'winning_submission.zip' is ready for download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
import os
from google.colab import files

# Define the exact names of the files we created earlier
model_path = 'best_model.pth'
submission_path = 'winning_submission.zip'

print("Preparing downloads...")

# 1. Download the Model Weights
if os.path.exists(model_path):
    print(f" Downloading your trained model: {model_path}")
    files.download(model_path)
else:
    print(f" Could not find {model_path}. Make sure you ran the training cell (Cell 5)!")

# 2. Download the Submission Zip
if os.path.exists(submission_path):
    print(f" Downloading your submission file: {submission_path}")
    files.download(submission_path)
else:
    print(f" Could not find {submission_path}. Make sure you ran the TTA inference cell (Cell 6)!")

Preparing downloads...
📥 Downloading your trained model: best_model.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Downloading your submission file: winning_submission.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>